# 03 モンテカルロシミュレーション

乱数で決まる行動・状態遷移・外部ショックを含むシミュレーションを**何回もくり返し**、平均とばらつきを求めます。
計算：`simulation.monte_carlo`

> **補遺トピック G2・教育用の抽象的な仮想モデル**です。国家（Country A〜E）・利得行列・状態遷移確率・係数はすべて架空で、現実の国際関係・紛争や特定国家の行動を予測するものではありません。扱うのは協調・対話・信頼形成・緊張緩和だけです。

## 準備：実験環境（最初に必ず実行）

G2 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（G2 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | G2 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（G2 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                         # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "g2-conflict-cooperation-simulation"    # リポジトリの中の G2 フォルダ（ルートが G2 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_g2_root(p):
    return (Path(p) / "game_theory.py").exists() and (Path(p) / "network_model.py").exists()


def find_g2_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または G2 のルート
        if is_g2_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_g2_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("G2 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/g2_upload")
    for hit in glob.glob("/content/g2_upload/**/game_theory.py", recursive=True):
        if is_g2_root(Path(hit).parent):
            return Path(hit).parent
    return None


G2_ROOT = find_g2_root()
if G2_ROOT is None:
    raise RuntimeError("G2 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(G2_ROOT)
if str(G2_ROOT) not in sys.path:
    sys.path.insert(0, str(G2_ROOT))
if not (G2_ROOT / "textbook" / "fixed_stack.py").exists():
    print("⚠️ textbook/ がありません。network_model.py などの import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import agents as ag
import game_theory as gt
import trust_model as tm
import state_transitions as st
import simulation as sim
import network_model as nm
import cooperation_policies as cp
import evaluation as ev
import conflict_charts as cc

cc.setup_style()                        # 文字を大きくするなら cc.setup_style(1.3)
COUNTRIES = ag.load_countries()         # 架空の国家 A〜E
PARAMS = sim.load_parameters()          # シミュレーションのパラメータ（すべて架空）
M = gt.load_payoff_matrix()             # 利得行列（架空）
IDS = [c["id"] for c in COUNTRIES]
RUNS = PARAMS["monte_carlo_runs"]       # モンテカルロの反復回数
YEARS = PARAMS["simulation_years"]
LABELS = {s: gt.label(s) for s in ag.STRATEGIES}

SAVE_FIGURES = False                    # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", cc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame([{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]))
    except ImportError:
        print(ev.format_table(rows, columns))


SUMMARY_COLUMNS = [("name", "condition", "{}"), ("cooperation_rate", "cooperation", "{:.3f}"),
                   ("cooperation_rate_sd", "sd", "{:.3f}"), ("mutual_cooperation_rate", "mutual coop.", "{:.3f}"),
                   ("cooperation_state_rate", "Cooperation state", "{:.3f}"), ("tension_rate", "Tension", "{:.3f}"),
                   ("conflict_rate", "Conflict", "{:.3f}"), ("conflict_rate_sd", "sd", "{:.3f}")]

print("G2_ROOT:", G2_ROOT)
print("架空の国家:", ", ".join(f"{c['id']}({LABELS[c['strategy']]})" for c in COUNTRIES))
print(f"モンテカルロ {RUNS} 回 × {YEARS} 年、乱数シード {PARAMS['random_seed']}")

## 1. 5 国・リングネットワークで 100 回 × 50 年

同じ乱数シードなら、同じ結果が再現されます。

In [ ]:
EDGES = nm.make_network("ring", IDS)
mc = sim.monte_carlo(COUNTRIES, EDGES, PARAMS, M, RUNS, years=YEARS)
s = mc["summary"]
again = sim.monte_carlo(COUNTRIES, EDGES, PARAMS, M, RUNS, years=YEARS)
print("同じシードで再現できる:", again["summary"] == s)
table([{"name": "ring, 5 countries", **s}], SUMMARY_COLUMNS)
print(f"協調状態の平均継続 {s['cooperation_duration']:.2f} 年、緊張状態の平均継続 {s['tension_duration']:.2f} 年、"
      f"紛争状態が 1 回でも起きた試行の割合 {s['ever_conflict']:.2f}")

## 2. 協調率と紛争状態率の分布

In [ ]:
#@title 協調率・紛争状態率の分布
show(cc.plot_hist([r["cooperation_rate"] for r in mc["runs"]], f"Cooperation rate over {RUNS} runs (model)",
                  "cooperation rate (per run)", "#2a78d6"), "03_hist_coop")
show(cc.plot_hist([r["conflict_rate"] for r in mc["runs"]], f"Conflict-state rate over {RUNS} runs (model)",
                  "conflict-state rate (per run)", "#d03b3b"), "03_hist_conflict")

## 3. 年数と協調率（試行の平均）

In [ ]:
#@title 年数と協調率
ys = mc["yearly"]
show(cc.plot_lines([y["year"] for y in ys],
                   {"cooperation rate (actions)": [y["cooperation_rate"] for y in ys],
                    "Cooperation state": [y["cooperation_state_rate"] for y in ys],
                    "Tension state": [y["tension_rate"] for y in ys],
                    "Conflict state": [y["conflict_rate"] for y in ys]},
                   f"Yearly averages over {RUNS} runs (model)", "share",
                   colors={"cooperation rate (actions)": "#3d3c39", "Cooperation state": cc.STATE_COLORS["Cooperation"],
                           "Tension state": cc.STATE_COLORS["Tension"], "Conflict state": cc.STATE_COLORS["Conflict"]},
                   styles={"cooperation rate (actions)": "--"}), "03_yearly")

## 4. 試行回数と推定値

試行回数を増やすと、平均（推定値）は安定していきます。

In [ ]:
#@title 試行回数による推定値の変化
big = sim.monte_carlo(COUNTRIES, EDGES, PARAMS, M, 1000, years=YEARS)
show(cc.plot_running_mean([r["conflict_rate"] for r in big["runs"]],
                          "Estimate of the conflict-state rate vs number of runs (model)", "running mean"),
     "03_running_mean")
for n in [10, 30, 100, 300, 1000]:
    vals = [r["conflict_rate"] for r in big["runs"][:n]]
    m_ = sum(vals) / n
    sd = (sum((v - m_) ** 2 for v in vals) / (n - 1)) ** 0.5
    print(f"{n:5} 回: 平均 {m_:.4f}  標準誤差 {sd / n ** 0.5:.4f}")

## 5. 学生向け TODO

In [ ]:
# TODO 1：試行回数を変更する / TODO 2：乱数シードを変更する
for runs, seed in [(30, 1), (30, 2), (300, 1), (300, 2)]:
    s2 = sim.monte_carlo(COUNTRIES, EDGES, PARAMS, M, runs, seed=seed, years=YEARS)["summary"]
    print(f"runs {runs:4} seed {seed}: 紛争状態率 {s2['conflict_rate']:.4f}")
# TODO 3：外部ショック確率を変更する
for q in [0.0, 0.05, 0.1, 0.2]:
    p = sim.with_changes(PARAMS, **{"transition.external_shock_probability": q})
    s3 = sim.monte_carlo(COUNTRIES, EDGES, p, M, RUNS, years=YEARS)["summary"]
    print(f"shock {q:.2f}: Tension {s3['tension_rate']:.3f}  Conflict {s3['conflict_rate']:.3f}")

## 考察（チームで記入）

- **TODO**：何回試行すれば、結論が乱数シードによって変わらなくなるか
- **TODO**：平均だけでなく、ばらつき（分布）から何が分かるか
- **TODO**：外部ショックの確率は、協調状態と紛争状態のどちらに大きく影響したか

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第4章 スタック・キュー、第5章 再帰、第6章 単純挿入ソート）
- R. Axelrod, *The Evolution of Cooperation*, Basic Books, 1984（繰り返し囚人のジレンマと Tit-for-Tat）
- G2 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_countries.json`、`data/payoff_matrix.json`、`data/simulation_parameters.json`（この教材のための**架空のデータ**）